# Part C — LangChain RAG Conversational Assistant (Hugging Face Inference Providers)

**National Heritage Preservation Trust (NHPT) — Coursework 01, Machine Learning and Related Applications (NB627BSDS)**

Same RAG concept as before, but the LLM now runs on **Hugging Face Inference Providers** (`meta-llama/Llama-3.1-8B-Instruct`) via the OpenAI-compatible router instead of a local Ollama server — no local model download, no `ollama serve` process required. Everything else keeps the same shape:

1. One reference document per class loaded into a **Chroma** vector store
2. An uploaded photo run through the **Part B ResNet50 model** for a structured prediction
3. A grounded, cited answer generated by **Llama 3.1 8B Instruct** via the Hugging Face router
4. **Multi-turn memory** across a conversation
5. A relevance-threshold hallucination guard, plus handling for empty/low-confidence/error cases

**Prerequisites**
- `cv_classification.ipynb` has been run, so `models/best_resnet50.pth` and `models/model_metadata.json` exist
- A Hugging Face account with an **access token** that has Inference Providers permission: https://huggingface.co/settings/tokens
- The token is available as the environment variable `HF_TOKEN` (this notebook loads it from a `.env` file — see the setup cell below)
- `knowledge_base/` contains one `.txt` file per class — six real reference documents matching your six classes are already included

### Structure
1. Setup & Health Check
2. Configuration
3. Knowledge Base -> Chroma Vector Store (local embeddings)
4. Load the Part B CV Model
5. RAG Chain (retriever + prompt + HF-hosted LLM)
6. Conversation Memory
7. Hallucination Guard
8. CV -> LLM Integration Pipeline
9. Example Conversations
10. Error Handling Demos
11. Design Choices & Limitations


## 1. Setup & Health Check

The LLM call goes to `https://router.huggingface.co/v1` using the standard `openai` Python SDK — Hugging Face Inference Providers exposes an OpenAI-compatible Chat Completions API, so no separate HF-specific client library is needed for generation.

**Set your token via a `.env` file** (not pasted into the notebook). Create a file named `.env` in the same folder as this notebook containing:
```
HF_TOKEN=hf_your_real_token_here
```
Get a token (needs **"Make calls to Inference Providers"** permission) at: https://huggingface.co/settings/tokens. Because the token lives in `.env` (which you should add to `.gitignore`), this notebook is safe to share or push to GitHub as-is.

In [1]:
import os
from dotenv import load_dotenv

# ── Load HF_TOKEN from a local .env file ──────────────────────────────────
# Create a file named `.env` next to this notebook containing one line:
#     HF_TOKEN=hf_your_real_token_here
# This keeps the token out of the notebook itself, so it's safe to share
# or push to GitHub without leaking credentials.
load_dotenv()

HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    print("HF_TOKEN loaded from .env / environment.")
else:
    print("HF_TOKEN not found.")
    print('  Create a ".env" file in this folder with: HF_TOKEN=hf_your_real_token_here')
    print("  Get a token (needs 'Make calls to Inference Providers' permission) at:")
    print("  https://huggingface.co/settings/tokens")


HF_TOKEN loaded from .env / environment.


In [2]:
%pip install -r requirements.txt

  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.3-py3-none-any.whl.metadata (4.6 kB)
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.4 MB 621.2 kB/s eta 0:00:16
   --- ------------------------------------ 0.8/10.4 MB 762.0 kB/s eta 0:00:13
   --- ------------------------------------ 0.8/10.4 MB 762.0 kB/s eta 0:00:13
   --- ------------

In [1]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install openai langchain-openai langchain-huggingface langchain-chroma langchain-text-splitters langchain-core sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json
import time
import shutil
import warnings
from pathlib import Path

import numpy as np
from PIL import Image
from dotenv import load_dotenv

import torch
import torch.nn as nn
from torchvision import models, transforms

from openai import OpenAI

from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser

warnings.filterwarnings("ignore", category=UserWarning)
load_dotenv()
print("Imports OK")


c:\Users\abish\anaconda3\envs\torch_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [2]:
def check_hf_health():
    token = os.environ.get("HF_TOKEN")
    if not token or token == "hf_yourtokenhere":
        print("HF_TOKEN is not set.")
        print("  Paste your real token into the cell above (replacing 'hf_yourtokenhere'),")
        print("  then re-run that cell followed by this one.")
        print("  Get a token at https://huggingface.co/settings/tokens")
        print("  (needs 'Make calls to Inference Providers' permission)")
        return False

    try:
        client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=token)
        resp = client.chat.completions.create(
            model="meta-llama/Llama-3.1-8B-Instruct:novita",
            messages=[{"role": "user", "content": "Reply with exactly: ready"}],
            max_tokens=10,
        )
        print(f"HF Inference Providers reachable. Model replied: {resp.choices[0].message.content!r}")
        return True
    except Exception as e:
        print(f"HF Inference Providers call failed: {e}")
        print("  Check that: the token is valid, has Inference Providers access,")
        print("  and the model/provider ('novita') is currently available.")
        return False

hf_ready = check_hf_health()


HF Inference Providers reachable. Model replied: 'Ready.'


## 2. Configuration

In [3]:
RAG_CONFIG = {
    # LLM — Hugging Face Inference Providers, OpenAI-compatible endpoint
    "hf_base_url": "https://router.huggingface.co/v1",
    "llm_model": "meta-llama/Llama-3.1-8B-Instruct:novita",
    "llm_temperature": 0.3,          # low temperature -> factual, low-variance answers
    "llm_max_tokens": 600,

    # Embeddings — run locally so indexing doesn't depend on an external embedding
    # endpoint or add extra HF API usage. sentence-transformers/all-MiniLM-L6-v2 is
    # small, fast on CPU, and good enough for short heritage reference documents.
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",

    "chunk_size": 500,
    "chunk_overlap": 50,
    "retrieval_k": 4,
    "relevance_threshold": 0.3,      # below this similarity, treat retrieval as empty
    "low_confidence_threshold": 0.6, # below this CV confidence, the assistant hedges

    "knowledge_base_dir": "./knowledge_base",
    "chroma_db_dir": "./chroma_db",
    "model_path": "./models/best_resnet50.pth",
    "model_metadata_path": "./models/model_metadata.json",
}

for k, v in RAG_CONFIG.items():
    print(f"  {k}: {v}")


  hf_base_url: https://router.huggingface.co/v1
  llm_model: meta-llama/Llama-3.1-8B-Instruct:novita
  llm_temperature: 0.3
  llm_max_tokens: 600
  embedding_model: sentence-transformers/all-MiniLM-L6-v2
  chunk_size: 500
  chunk_overlap: 50
  retrieval_k: 4
  relevance_threshold: 0.3
  low_confidence_threshold: 0.6
  knowledge_base_dir: ./knowledge_base
  chroma_db_dir: ./chroma_db
  model_path: ./models/best_resnet50.pth
  model_metadata_path: ./models/model_metadata.json


## 3. Knowledge Base -> Chroma Vector Store

Loads every `.txt` file in `knowledge_base/`, splits it into overlapping chunks, embeds each chunk locally, and stores it in a persistent Chroma collection. Each chunk keeps `source` (filename) and `style` metadata so answers can cite exactly where they came from.

Six real reference documents are already included, one per class:

In [4]:
# ── Load every .txt file already in knowledge_base/ ────────────────────────
# This folder is the sole source of truth — no sample/default documents are
# generated or injected here. Add, edit, or remove .txt files directly in
# knowledge_base/ and re-run this cell onward to pick up the changes.
# Filename stem (without .txt) is used as the "style" label, so name each
# file to match a class, e.g. "Baroque architecture.txt".
KB_DIR = Path(RAG_CONFIG["knowledge_base_dir"])

if not KB_DIR.exists():
    raise FileNotFoundError(
        f"{KB_DIR}/ does not exist. Create it and add one .txt file per class."
    )

kb_documents = []
for file_path in sorted(KB_DIR.glob("*.txt")):
    text = file_path.read_text(encoding="utf-8").strip()
    style_name = file_path.stem
    kb_documents.append(Document(
        page_content=text,
        metadata={"source": file_path.name, "style": style_name},
    ))
    print(f"  Loaded {style_name}  ({len(text.split())} words)  <- {file_path}")

if not kb_documents:
    raise RuntimeError(f"No .txt files found in {KB_DIR}/ — add at least one document.")

print(f"\nLoaded {len(kb_documents)} knowledge base documents from {KB_DIR}/")


  Loaded Achaemenid architecture  (94 words)  <- knowledge_base\Achaemenid architecture.txt
  Loaded Baroque architecture  (635 words)  <- knowledge_base\Baroque architecture.txt
  Loaded Deconstructivism  (659 words)  <- knowledge_base\Deconstructivism.txt
  Loaded Novelty architecture  (630 words)  <- knowledge_base\Novelty architecture.txt
  Loaded Queen Anne architecture  (617 words)  <- knowledge_base\Queen Anne architecture.txt
  Loaded Tudor Revival architecture  (657 words)  <- knowledge_base\Tudor Revival architecture.txt

Loaded 6 knowledge base documents from knowledge_base/


In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=RAG_CONFIG["chunk_size"],
    chunk_overlap=RAG_CONFIG["chunk_overlap"],
)

all_chunks = splitter.split_documents(kb_documents)
print(f"Split {len(kb_documents)} documents into {len(all_chunks)} chunks")


Split 6 documents into 79 chunks


In [6]:
embeddings = HuggingFaceEmbeddings(model_name=RAG_CONFIG["embedding_model"])
print(f"Local embedding model ready: {RAG_CONFIG['embedding_model']}")

chroma_path = RAG_CONFIG["chroma_db_dir"]
collection_name = "nhpt_heritage"

# Release any handle left over from a previous run in this same kernel session
# before touching the DB on disk.
try:
    del vectorstore
except NameError:
    pass
import gc
gc.collect()

# Clear the collection through the Chroma client API instead of deleting the
# folder on disk. Deleting the folder (shutil.rmtree) can raise a
# PermissionError on Windows if a previous run's SQLite connection is still
# open — clearing via the client avoids that file lock entirely.
import chromadb

if os.path.exists(chroma_path):
    try:
        client = chromadb.PersistentClient(path=chroma_path)
        existing = [c.name for c in client.list_collections()]
        if collection_name in existing:
            client.delete_collection(collection_name)
            print("Cleared existing Chroma collection (rebuilding fresh)")
        del client
    except Exception as e:
        print(f"Could not clear existing collection cleanly ({e}); continuing anyway.")

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory=chroma_path,
    collection_name=collection_name,
)
print(f"Chroma vector store built: {len(all_chunks)} chunks -> {chroma_path}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1711.46it/s]


Local embedding model ready: sentence-transformers/all-MiniLM-L6-v2
Cleared existing Chroma collection (rebuilding fresh)
Chroma vector store built: 79 chunks -> ./chroma_db


In [7]:
# ── Sanity check: does retrieval actually find relevant chunks? ──────────
test_query = "What kind of columns and roof does this style use?"
results = vectorstore.similarity_search_with_relevance_scores(test_query, k=3)

print(f"Test query: '{test_query}'\n")
for i, (doc, score) in enumerate(results):
    print(f"Result {i+1}  (score={score:.3f})  style={doc.metadata['style']}  source={doc.metadata['source']}")
    print(f"  {doc.page_content[:140]}...\n")


Test query: 'What kind of columns and roof does this style use?'

Result 1  (score=0.417)  style=Queen Anne architecture  source=Queen Anne architecture.txt
  * Round or Polygonal Towers: Corner turrets with conical "witch's hat" roofs are a signature feature, giving homes a castle-like appearance....

Result 2  (score=0.346)  style=Tudor Revival architecture  source=Tudor Revival architecture.txt
  * Decorative Half-Timbering: Grid-like patterns of dark wood contrast sharply against white or cream stucco walls. [16, 17, 18, 19, 20] 
* M...

Result 3  (score=0.301)  style=Tudor Revival architecture  source=Tudor Revival architecture.txt
  When the style traveled to the United States and Canada after World War I, it evolved significantly. Returning soldiers who had fallen in lo...



## 4. Load the Part B CV Model

Reconstructs the ResNet50 classifier architecture from `cv_classification.ipynb`, loads the trained weights, and rebuilds the `predict_style()` function from the saved metadata — no retraining needed.

In [8]:
with open(RAG_CONFIG["model_metadata_path"]) as f:
    model_metadata = json.load(f)

CLASS_NAMES = model_metadata["class_names"]
NUM_CLASSES = model_metadata["num_classes"]
IMG_SIZE = model_metadata["img_size"]
IMAGENET_MEAN = model_metadata["imagenet_mean"]
IMAGENET_STD = model_metadata["imagenet_std"]

print(f"Model metadata loaded: {NUM_CLASSES} classes -> {CLASS_NAMES}")

kb_styles = {doc.metadata["style"] for doc in kb_documents}


Model metadata loaded: 6 classes -> ['Achaemenid architecture', 'Baroque architecture', 'Deconstructivism', 'Novelty architecture', 'Queen Anne architecture', 'Tudor Revival architecture']


In [9]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def build_model(num_classes):
    m = models.resnet50(weights=None)  # weights loaded from checkpoint below
    in_features = m.fc.in_features
    m.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.2),
        nn.Linear(256, num_classes),
    )
    return m

cv_model = build_model(NUM_CLASSES)
cv_model.load_state_dict(torch.load(RAG_CONFIG["model_path"], map_location=DEVICE))
cv_model.to(DEVICE).eval()

cv_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print(f"CV model loaded on {DEVICE} — ready for inference")


CV model loaded on cuda — ready for inference


In [10]:
def predict_style(img_path, top_k=3):
    """Run the CV model on an image and return a JSON-ready structured prediction."""
    img = Image.open(img_path).convert("RGB")
    tensor = cv_transform(img).unsqueeze(0).to(DEVICE)

    start = time.time()
    with torch.no_grad():
        probs = torch.softmax(cv_model(tensor), dim=1).cpu().numpy()[0]
    elapsed_ms = (time.time() - start) * 1000

    top_idx = probs.argsort()[::-1][:top_k]
    return {
        "predicted_style": CLASS_NAMES[top_idx[0]],
        "confidence": float(probs[top_idx[0]]),
        "top_k": [(CLASS_NAMES[i], float(probs[i])) for i in top_idx],
        "inference_time_ms": round(elapsed_ms, 2),
    }


## 5. RAG Chain

**Why `ChatOpenAI` for a Hugging Face model?** HF Inference Providers exposes an OpenAI-compatible Chat Completions API, and LangChain's `ChatOpenAI` class accepts any `base_url` — pointing it at `https://router.huggingface.co/v1` gets a full LangChain-native chat model (usable in prompt chains, with streaming, etc.) backed by `meta-llama/Llama-3.1-8B-Instruct`, without writing a custom LangChain wrapper.

**Prompt design.** Same as before: the system prompt restricts the model to the provided context, requires source citation, and gives one fixed fallback line for out-of-scope questions. When a CV prediction is available, its confidence is injected into the prompt so the model hedges automatically on uncertain predictions.

In [11]:
llm = ChatOpenAI(
    base_url=RAG_CONFIG["hf_base_url"],
    api_key=os.environ["HF_TOKEN"],
    model=RAG_CONFIG["llm_model"],
    temperature=RAG_CONFIG["llm_temperature"],
    max_tokens=RAG_CONFIG["llm_max_tokens"],
)

test = llm.invoke("Reply with exactly: ready")
print(f"LLM check: {test.content}")


LLM check: Ready


In [12]:
SYSTEM_PROMPT = """You are a knowledgeable heritage guide for the National Heritage Preservation Trust (NHPT).
You help visitors understand the architectural styles and artifacts at historic sites.

RULES:
1. Answer ONLY using the CONTEXT below. Do not rely on outside knowledge.
2. Always name the source document you drew from (e.g. "based on the source on {{style}}...").
3. If the context does not contain enough information, reply exactly:
   "I don't have enough information in my knowledge base to answer that question."
4. Keep answers to 2-4 short paragraphs.
5. If a CV confidence score is given below 60%, explicitly flag the uncertainty and mention
   the next most likely style before answering.

CONTEXT:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

print("Prompt template ready")

Prompt template ready


In [13]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": RAG_CONFIG["retrieval_k"]},
)

def format_context(docs):
    if not docs:
        return "(no relevant context retrieved)"
    return "\n\n".join(f"[Source: {d.metadata['source']}]\n{d.page_content}" for d in docs)

rag_chain = rag_prompt | llm | StrOutputParser()

def answer(question, chat_history, cv_context=None):
    """Retrieve, build context (optionally prefixed with a CV prediction), and generate."""
    docs_and_scores = vectorstore.similarity_search_with_relevance_scores(
        question, k=RAG_CONFIG["retrieval_k"]
    )
    relevant = [d for d, s in docs_and_scores if s >= RAG_CONFIG["relevance_threshold"]]

    context_text = format_context(relevant)
    if cv_context:
        context_text = f"[CV Prediction]\n{json.dumps(cv_context, indent=2)}\n\n{context_text}"

    response = rag_chain.invoke({
        "context": context_text,
        "chat_history": chat_history,
        "input": question,
    })
    return response, relevant

print("RAG answer() function ready: retriever -> context stuffing -> HF-hosted LLM")

RAG answer() function ready: retriever -> context stuffing -> HF-hosted LLM


## 6. Conversation Memory

`InMemoryChatMessageHistory` per session id, so the assistant remembers earlier turns within a conversation (the multi-turn requirement) without persisting anything to disk between kernel runs.

In [14]:
session_store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]


def chat(session_id, question, cv_context=None, verbose=True):
    history = get_session_history(session_id)
    response, sources = answer(question, history.messages, cv_context=cv_context)

    history.add_user_message(question)
    history.add_ai_message(response)

    if verbose:
        print(f"[{session_id}] USER: {question}")
        if cv_context:
            print(f"    (CV: {cv_context['predicted_style']} @ {cv_context['confidence']:.0%})")
        print(f"[{session_id}] ASSISTANT: {response}\n")
    return response, sources

print("chat() ready — pass a session_id to keep multi-turn history")


chat() ready — pass a session_id to keep multi-turn history


## 7. Hallucination Guard

Two layers, deliberately redundant: the prompt tells the LLM to refuse when context is thin, and the code independently drops any retrieved chunk below `relevance_threshold` *before* the LLM ever sees it. The code-level guard is the one that actually matters — it works even if the hosted model doesn't perfectly follow its instructions.

In [15]:
def relevance_guard_demo(question):
    docs_and_scores = vectorstore.similarity_search_with_relevance_scores(
        question, k=RAG_CONFIG["retrieval_k"]
    )
    kept = [(d, s) for d, s in docs_and_scores if s >= RAG_CONFIG["relevance_threshold"]]
    print(f"Query: '{question}'")
    print(f"  Retrieved: {len(docs_and_scores)}  |  Above threshold ({RAG_CONFIG['relevance_threshold']}): {len(kept)}")
    for d, s in docs_and_scores:
        flag = "KEEP" if s >= RAG_CONFIG["relevance_threshold"] else "DROP"
        print(f"    [{flag}] score={s:.3f}  source={d.metadata['source']}")
    return kept

# An in-scope query should keep most chunks; an unrelated one should drop them all.
_ = relevance_guard_demo("What materials and structural features define this style?")
print()
_ = relevance_guard_demo("What's the best recipe for chocolate cake?")


Query: 'What materials and structural features define this style?'
  Retrieved: 4  |  Above threshold (0.3): 1
    [KEEP] score=0.304  source=Deconstructivism.txt
    [DROP] score=0.293  source=Novelty architecture.txt
    [DROP] score=0.284  source=Queen Anne architecture.txt
    [DROP] score=0.282  source=Deconstructivism.txt

Query: 'What's the best recipe for chocolate cake?'
  Retrieved: 4  |  Above threshold (0.3): 0
    [DROP] score=-0.110  source=Baroque architecture.txt
    [DROP] score=-0.213  source=Tudor Revival architecture.txt
    [DROP] score=-0.247  source=Novelty architecture.txt
    [DROP] score=-0.259  source=Baroque architecture.txt


## 8. CV -> LLM Integration Pipeline

The handoff contract is a plain JSON dict (`predicted_style`, `confidence`, `top_k`, `inference_time_ms`) produced by `predict_style()`. The RAG layer never touches CV internals — it just reads this dict, so the two systems can be developed and tested independently.

In [16]:
def visual_query(session_id, img_path, question="What can you tell me about this?"):
    prediction = predict_style(img_path)
    return chat(session_id, question, cv_context=prediction)

print("visual_query() ready: image path -> CV prediction -> grounded RAG answer")


visual_query() ready: image path -> CV prediction -> grounded RAG answer


In [17]:
# ── Integration test: run one image per class through the full pipeline ──
data_dir = Path("./data")
print("INTEGRATION TEST: CV -> LLM across all classes")
print("=" * 60)

times = []
for class_dir in sorted(p for p in data_dir.iterdir() if p.is_dir()):
    images = [f for f in class_dir.iterdir() if f.suffix.lower() in (".jpg", ".jpeg", ".png")]
    if not images:
        print(f"  (skipping {class_dir.name} — no images found)")
        continue
    img_path = images[0]
    try:
        pred = predict_style(str(img_path))
        times.append(pred["inference_time_ms"])
        print(f"  {class_dir.name:30s} -> {pred['predicted_style']:30s} ({pred['confidence']:.0%})")
    except Exception as e:
        print(f"  {class_dir.name:30s} -> ERROR: {e}")

if times:
    print(f"\nAverage CV inference time: {np.mean(times):.1f} ms")


INTEGRATION TEST: CV -> LLM across all classes
  Achaemenid architecture        -> Achaemenid architecture        (100%)
  Baroque architecture           -> Baroque architecture           (100%)
  Deconstructivism               -> Deconstructivism               (99%)
  Novelty architecture           -> Novelty architecture           (100%)
  Queen Anne architecture        -> Queen Anne architecture        (98%)
  Tudor Revival architecture     -> Tudor Revival architecture     (100%)

Average CV inference time: 143.5 ms


## 9. Example Conversations

Demonstrates the required conversation types: a high-confidence visual query, a multi-turn follow-up (3+ exchanges), a text-only knowledge question, and a low-confidence CV case where the assistant should hedge.

Set `SAMPLE_IMAGE` to a real file under `data/` to run the visual demos.

In [18]:
from pathlib import Path

data_dir = Path("./data")
for class_dir in sorted(p for p in data_dir.iterdir() if p.is_dir()):
    images = [f for f in class_dir.iterdir() if f.suffix.lower() in (".jpg", ".jpeg", ".png")]
    if images:
        print(images[0])

data\Achaemenid architecture\000000.jpg
data\Baroque architecture\003211.jpg
data\Deconstructivism\006918.jpg
data\Novelty architecture\010434.jpg
data\Queen Anne architecture\012200.jpg
data\Tudor Revival architecture\014059.jpg


In [19]:
SAMPLE_IMAGE = "data/Baroque architecture/103_800px-Old_Goa_Church_02.jpg"

if SAMPLE_IMAGE and Path(SAMPLE_IMAGE).exists():
    print("--- Conversation 1: high-confidence visual query ---")
    visual_query("conv1", SAMPLE_IMAGE, "What style is this and what defines it?")

    print("--- Conversation 2: multi-turn follow-up ---")
    chat("conv2", "Tell me about the style shown in this photo.",
         cv_context=predict_style(SAMPLE_IMAGE))
    chat("conv2", "What time period does that date from?")
    chat("conv2", "How does it compare to more modern styles?")
else:
    print("Set SAMPLE_IMAGE to a real file path under data/ to run the visual conversation demos.")

print("--- Conversation 3: text-only knowledge question (no image) ---")
chat("conv3", "What construction materials are typical of these heritage styles?")

--- Conversation 1: high-confidence visual query ---
[conv1] USER: What style is this and what defines it?
    (CV: Baroque architecture @ 100%)
[conv1] ASSISTANT: Based on the source, the predicted style is "Baroque architecture" with a high confidence score of 99.93%. 

Baroque architecture is a style that originated in Europe in the 17th century. It is characterized by dramatic lighting, intense emotions, and highly ornamented decoration. This style often features sweeping curves, intricate details, and a sense of movement and energy. Baroque architecture often incorporates elements such as arches, domes, and grandiose proportions to create a sense of awe and drama.

The term "Baroque" comes from the Portuguese word for "irregularly shaped pearl." This style emerged as a response to the more restrained and classical styles of the Renaissance, and it is often associated with the Catholic Church and the monarchies of Europe.

--- Conversation 2: multi-turn follow-up ---
[conv2] USER: 

('Based on the provided sources, the typical construction materials for these heritage styles are as follows:\n\n* Deconstructivism: Heavy use of raw, reflective materials like titanium, zinc, poured concrete, and chain-link fencing. (Source: Deconstructivism.txt)\n* Queen Anne architecture: Walls combine smooth brick, patterned wooden shingles (like fish-scale or diamond shapes), and decorative stone. (Source: Queen Anne architecture.txt)\n* Tudor Revival architecture: Decorative half-timbering uses dark wood, while walls are often made of white or cream stucco. Massive brick chimneys are also a feature. (Source: Tudor Revival architecture.txt)\n\nThese materials contribute to the unique character and aesthetic of each style.',
 [Document(id='57147de5-fbb8-4cf7-a672-0640ef285541', metadata={'style': 'Deconstructivism', 'source': 'Deconstructivism.txt'}, page_content="* Exploded Forms: Buildings look like a collection of separate, jagged pieces colliding into one another.\n* Absence of

In [20]:
print("--- Conversation 4: low-confidence CV case (simulated) ---")
fake_low_confidence_prediction = {
    "predicted_style": CLASS_NAMES[0],
    "confidence": 0.42,
    "top_k": [(CLASS_NAMES[0], 0.42), (CLASS_NAMES[1], 0.35), (CLASS_NAMES[2], 0.23)],
    "inference_time_ms": 18.4,
}
chat("conv4", "What can you tell me about this building?", cv_context=fake_low_confidence_prediction)

print("--- Conversation 5: out-of-scope question (hallucination guard) ---")
chat("conv5", "What's the weather like in Colombo today?")


--- Conversation 4: low-confidence CV case (simulated) ---
[conv4] USER: What can you tell me about this building?
    (CV: Achaemenid architecture @ 42%)
[conv4] ASSISTANT: Based on the provided context, I don't have enough information in my knowledge base to answer that question about the building. However, I can tell you that the predicted style of the building is Achaemenid architecture with a confidence score of 42%. 

If I had to consider other possibilities, the next most likely style would be Baroque architecture, which is a style characterized by grandeur, ornamentation, and dramatic lighting. However, without more information about the building, I can only speculate about its architectural style.

If you could provide more details or context about the building, I would be happy to try and assist you further.

--- Conversation 5: out-of-scope question (hallucination guard) ---
[conv5] USER: What's the weather like in Colombo today?
[conv5] ASSISTANT: I don't have enough inform

("I don't have enough information in my knowledge base to answer that question.",
 [])

## 10. Error Handling Demos

In [21]:
print("--- Invalid / corrupted image path ---")
try:
    predict_style("./data/does_not_exist.jpg")
except FileNotFoundError as e:
    print(f"  Handled cleanly: {e}")

print("\n--- Empty / irrelevant retrieval ---")
response, sources = answer("Tell me about 20th century car engines.", [])
print(f"  Sources kept above threshold: {len(sources)}")
print(f"  Response: {response}")

print("\n--- Hugging Face endpoint unreachable / rate-limited (simulated) ---")
print("  If HF_TOKEN is missing, invalid, or the provider is temporarily unavailable,")
print("  llm.invoke() raises an exception from the openai client (e.g. AuthenticationError,")
print("  RateLimitError, or APIConnectionError). Wrap chat()/answer() calls in try/except")
print("  and surface the same fix instructions as the Section 1 health check to recover gracefully.")

def safe_chat(session_id, question, cv_context=None):
    try:
        return chat(session_id, question, cv_context=cv_context)
    except Exception as e:
        print(f"  LLM call failed ({type(e).__name__}): {e}")
        print("  Falling back to a safe default response.")
        return "I'm having trouble reaching the heritage assistant right now. Please try again shortly.", []

print("\nsafe_chat() available as a drop-in wrapper for production use.")


--- Invalid / corrupted image path ---
  Handled cleanly: [Errno 2] No such file or directory: './data/does_not_exist.jpg'

--- Empty / irrelevant retrieval ---
  Sources kept above threshold: 0
  Response: I don't have enough information in my knowledge base to answer that question.

--- Hugging Face endpoint unreachable / rate-limited (simulated) ---
  If HF_TOKEN is missing, invalid, or the provider is temporarily unavailable,
  llm.invoke() raises an exception from the openai client (e.g. AuthenticationError,
  RateLimitError, or APIConnectionError). Wrap chat()/answer() calls in try/except
  and surface the same fix instructions as the Section 1 health check to recover gracefully.

safe_chat() available as a drop-in wrapper for production use.


## 11. Design Choices & Limitations (~500 words)

**Why Hugging Face Inference Providers instead of a local Ollama model.** Routing through `https://router.huggingface.co/v1` with the standard `openai` SDK gives access to `meta-llama/Llama-3.1-8B-Instruct` (an 8-billion-parameter model) without downloading multi-gigabyte weights or running a local inference server, which matters for a coursework machine that may not have a capable GPU. The trade-off is a network dependency: every LLM call now needs internet access and a valid `HF_TOKEN`, and is subject to the provider's latency and (for the free tier) rate limits, whereas Ollama's local Llama 3.2 had zero network dependency once pulled. The health check in Section 1 exists specifically to surface that dependency early rather than letting it fail deep inside a chain.

**Why `ChatOpenAI` rather than a bespoke HF wrapper.** Hugging Face Inference Providers deliberately exposes an OpenAI-compatible Chat Completions API, so LangChain's existing `ChatOpenAI` class works unmodified by just pointing `base_url` at the HF router — this keeps the RAG pipeline fully LangChain-native (usable inside `Runnable` chains, with the same prompt templates and output parsers) rather than writing custom glue code around the raw `openai` client, and it means swapping to a different OpenAI-compatible provider later is a one-line change.

**Embeddings stay local.** Rather than also calling an HF-hosted embedding endpoint for every chunk during indexing, embeddings use `sentence-transformers/all-MiniLM-L6-v2` running on CPU via `langchain_huggingface.HuggingFaceEmbeddings`. This keeps indexing fast, free, and offline-capable — appropriate for a knowledge base of six short documents that gets rebuilt from scratch on every run — and avoids burning API quota on embedding calls that don't need the generative model's capabilities.

**Prompt design and low temperature (0.3) are unchanged from the Ollama version:** the system prompt is deliberately blunt, restricting the model to supplied context, requiring source citation, and giving one fixed fallback sentence for out-of-scope questions, rather than trusting a softer instruction. Even an 8B instruction-tuned hosted model can drift into its own parametric knowledge, so the phrasing stays a hard rule rather than a suggestion.

**Hallucination mitigation** keeps the same two-layer design: a prompt-level instruction, and a code-level relevance-score filter that drops any retrieved chunk below 0.3 similarity before it reaches the LLM at all. The code-level guard is what actually guarantees the "I don't know" behaviour, since it doesn't depend on the model choosing to follow instructions — Section 7 demonstrates it filtering out chunks on an unrelated query regardless of what the LLM would have said.

**CV-LLM handoff** is unchanged: a fixed JSON contract (`predicted_style`, `confidence`, `top_k`, `inference_time_ms`) from `predict_style()`, consumed as-is by the RAG layer, so swapping the LLM provider required zero changes to the CV integration code — only Sections 1 and 5 (health check and chain construction) differ from the Ollama version.

**Known limitations.** Every generation call now costs latency and, beyond free-tier limits, money — unsuitable for very high query volume without caching or a paid tier. The `:novita` provider suffix pins a specific backing provider for the model; if that provider becomes unavailable, the model string needs updating (Hugging Face's router page lists current provider options per model). Memory remains in-process only (`InMemoryChatMessageHistory`), lost on kernel restart — acceptable for a coursework prototype, but a production deployment would need a persistent store.
